In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read data

In [3]:
# read report in xlsx format
report_tumor = pd.read_excel(
    'article/41467_2023_39981_MOESM7_ESM.xlsx',
    sheet_name=1, index_col=0, 
    )
report_normal = pd.read_excel(
    'article/41467_2023_39981_MOESM7_ESM.xlsx',
    sheet_name=2, index_col=0
    )

In [4]:
# skip the first column
report_tumor = report_tumor.iloc[:, 1:]
report_normal = report_normal.iloc[:, 1:]

In [5]:
# merge two reports using index
report = pd.concat([report_tumor, report_normal], axis=1)

In [6]:
metadata = pd.read_excel(
    'article/41467_2023_39981_MOESM4_ESM.xlsx',
    sheet_name=1
    )
tumor_meta = metadata[['Tumor No.']].copy()
tumor_meta['Condition'] = 'Tumor'
normal_meta = metadata[['Adjacent No.']].copy()
normal_meta['Condition'] = 'Normal'
# rename columns - Tumor No. and Adjacent No. to sample
tumor_meta.columns = ['Sample', 'Condition']
normal_meta.columns = ['Sample', 'Condition']

# merge two metadata
metadata = pd.concat([tumor_meta, normal_meta], axis=0).reset_index(drop=True)
metadata['Dataset'] = 'PXD042844'
# exclude samples that are not in the report colnames (and print them)
print(metadata[~metadata['Sample'].isin(report.columns)])
metadata = metadata[metadata['Sample'].isin(report.columns)]

        Sample Condition    Dataset
229  Exo057104    Normal  PXD042844


In [7]:
# order report by metadata sample
print(f"Shape report: {report.shape}")
report = report.loc[:, metadata['Sample']]
print(f"Shape report: {report.shape}")

Shape report: (12310, 230)
Shape report: (12310, 229)


In [11]:
# remove rows with all NAs in report
report = report.dropna(how='all')
print(f"Shape report: {report.shape}")

Shape report: (12304, 229)


In [14]:
# check if non-inque values are in index of report
print(report.index.duplicated().sum())
# print which one
print(report.index[report.index.duplicated()])
# remove one with smaller values or more missing values
report = report[~report.index.duplicated(keep='first')]
print(f"Shape report: {report.shape}")

1
Index([43891], dtype='object', name='Symbol')
Shape report: (12303, 229)


In [15]:
# save report to csv
report.to_csv('report.csv', index="Gene", sep='\t')
# save metadata to csv
metadata.to_csv('metadata.csv', index=False, sep='\t')

In [16]:
print(f"report shape: {report.shape}")
print(f"metadata shape: {metadata.shape}")

report shape: (12303, 229)
metadata shape: (229, 3)
